In [3]:
import polars as pl

# batch 8

In [4]:
import extern
extern.run("grep SRA submit_batch3 |awk '{print $2}' |cat - runlists/batch1_500.txt runlists/batch2_500.txt runlists/batch4_2000.txt runlists/batch5_test10.txt runlists/batch6_test50000.txt runlists/batch7_test10000.txt >/tmp/runs")

''

In [5]:
prev = pl.read_csv("/tmp/runs", has_header=False)
prev.columns = ["acc"]
prev.shape

(63406, 1)

In [6]:
total = pl.read_csv('runlists/acc_less_than_20k_mbytes.csv', has_header=False)
total.columns = ["acc"]
total.shape

(710928, 1)

In [7]:
batch8_possible = total.join(prev, on="acc", how="anti")
batch8_possible.shape

(647522, 1)

In [8]:
# batch8 = batch8_possible.sample(100000)
# print(batch8.shape, batch8[:5])
# batch8.write_csv('runlists/batch8_100000.txt', include_header=False)

In [9]:
# 
extern.run("grep SRA submit_batch3 |awk '{print $2}' |cat - runlists/batch1_500.txt runlists/batch2_500.txt runlists/batch4_2000.txt runlists/batch5_test10.txt runlists/batch6_test50000.txt runlists/batch7_test10000.txt runlists/batch8_100000.txt >/tmp/runs")

prev = pl.read_csv("/tmp/runs", has_header=False)
prev.columns = ["acc"]
print(prev.shape)

batch9_possible = total.join(prev, on="acc", how="anti")
batch9_possible.shape

(163406, 1)


(547522, 1)

In [10]:
# batch9_possible.write_csv('runlists/batch9_da_rest.txt', include_header=False)

# batch10 - cleaning up and everythin 10Gbp or less

In [11]:
extern.run('cat  s3_us-east2.txt s3_woodcrob-sandpiper-us-east-1.txt |grep RR |sed "s/  */\t/g" >/tmp/runs')

prev1 = pl.read_csv('/tmp/runs', has_header=False, separator='\t')
prev1.columns = ["acc", "time", "size", "path"]
prev2 = prev1.with_columns(pl.col("path").alias("acc").str.split('.').list.get(0)).select(["acc", "size"])
prev2.shape, prev2[:3]

((137852, 2),
 shape: (3, 2)
 ┌───────────┬───────┐
 │ acc       ┆ size  │
 │ ---       ┆ ---   │
 │ str       ┆ i64   │
 ╞═══════════╪═══════╡
 │ DRR001961 ┆ 26759 │
 │ DRR003630 ┆ 60975 │
 │ DRR003636 ┆ 45914 │
 └───────────┴───────┘)

In [12]:
prev2.filter(pl.col("size") == 0).shape

(4305, 2)

In [13]:
prev3 = prev2.filter(pl.col("size") > 0)
prev3.shape

(133547, 2)

In [14]:
df = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20250220.some_columns.csv.gz', has_header=False)
df.columns = ['acc','releasedate','mbases','mbytes']
df[:4]

acc,releasedate,mbases,mbytes
str,str,i64,i64
"""SRR15442735""","""2021-08-13T00:00:00+00:00""",6638,2614
"""ERR1959224""","""2017-07-08T00:00:00+00:00""",8555,3195
"""ERR5003368""","""2020-12-23T00:00:00+00:00""",1013,344
"""ERR5261058""","""2021-03-15T00:00:00+00:00""",20619,6792


In [15]:
batch10_possible = df.filter(pl.col('mbases') < 8000).join(prev, on="acc", how="anti")
batch10_possible.shape, batch10_possible.sample(3)

((471178, 4),
 shape: (3, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ SRR22519752 ┆ 2022-12-04T00:00:00+00:00 ┆ 5334   ┆ 1712   │
 │ SRR25435948 ┆ 2024-03-20T00:00:00+00:00 ┆ 1075   ┆ 322    │
 │ SRR19917995 ┆ 2024-06-17T00:00:00+00:00 ┆ 1998   ┆ 849    │
 └─────────────┴───────────────────────────┴────────┴────────┘)

In [16]:
batch10_possible.select('acc').write_csv('runlists/batch10_10gbp_or_smaller.txt', include_header=False)

# batch11 small fast test of multiple

In [17]:
per_acc = pl.read_csv('~/m/msingle/mess/174_R220_renew/processing_20240531/per_acc_summary.csv')
per_acc.shape, per_acc[:3]

((245436, 11),
 shape: (3, 11)
 ┌────────────┬───────────┬───────────┬───────────┬───┬─────────┬───────────┬───────────┬───────────┐
 │ sample     ┆ root_cove ┆ species_c ┆ bacterial ┆ … ┆ warning ┆ predictio ┆ organism  ┆ host_or_n │
 │ ---        ┆ rage      ┆ overage   ┆ _archaeal ┆   ┆ ---     ┆ n         ┆ ---       ┆ ot        │
 │ str        ┆ ---       ┆ ---       ┆ _bases    ┆   ┆ str     ┆ ---       ┆ str       ┆ ---       │
 │            ┆ f64       ┆ f64       ┆ ---       ┆   ┆         ┆ str       ┆           ┆ str       │
 │            ┆           ┆           ┆ i64       ┆   ┆         ┆           ┆           ┆           │
 ╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═════════╪═══════════╪═══════════╪═══════════╡
 │ SRR8634435 ┆ 345.4     ┆ 0.933237  ┆ 117232064 ┆ … ┆ null    ┆ host      ┆ feces met ┆ host      │
 │            ┆           ┆           ┆ 2         ┆   ┆         ┆           ┆ agenome   ┆           │
 │ SRR8640623 ┆ 730.5     ┆ 0.127748  ┆ 139594647 ┆

In [18]:
batch11 = batch10_possible.join(per_acc.filter(pl.col('root_coverage')>0), left_on='acc', right_on='sample', how='inner').sort('mbases').head(4)

In [19]:
# batch11.select('acc').write_csv('runlists/batch11_4.txt', include_header=False)

# batch 12 - 8 to 30Gbp

In [20]:
df = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20250220.some_columns.csv.gz', has_header=False)
df.columns = ['acc','releasedate','mbases','mbytes']
df[:4]

acc,releasedate,mbases,mbytes
str,str,i64,i64
"""SRR15442735""","""2021-08-13T00:00:00+00:00""",6638,2614
"""ERR1959224""","""2017-07-08T00:00:00+00:00""",8555,3195
"""ERR5003368""","""2020-12-23T00:00:00+00:00""",1013,344
"""ERR5261058""","""2021-03-15T00:00:00+00:00""",20619,6792


In [21]:
# how much 0-8, 8-15, 15-30 and 30+ mbases?
(
    df.select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 8000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 15000).filter(pl.col('mbases') >= 8000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 30000).filter(pl.col('mbases') >= 15000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') >= 30000).select('mbases').sum()[0,0] /1e9,
)


(4.866907013, 1.763913074, 1.36664454, 0.898161387, 0.838188012)

In [22]:
# Try those < 8gbases again where they failed - maybe the increased RAM of a this batch will get them through?
extern.run('cat s3_ls/* |grep RR |sed "s/  */\t/g" >/tmp/runs')

prev4 = pl.read_csv('/tmp/runs', has_header=False, separator='\t')
prev4.columns = ["acc", "time", "size", "path"]
print(prev4.group_by(pl.col('size') > 0).len())
prev5 = prev4.filter(pl.col('size') > 0).with_columns(pl.col("path").alias("acc").str.split('.').list.get(0))
prev5.shape

shape: (2, 2)
┌───────┬────────┐
│ size  ┆ len    │
│ ---   ┆ ---    │
│ bool  ┆ u32    │
╞═══════╪════════╡
│ false ┆ 8769   │
│ true  ┆ 591446 │
└───────┴────────┘


(591446, 4)

In [23]:
# Create a batch of 1000 runs with 8-30 Gbps, that aren't in any previous 
batch12_possible = df.filter(pl.col('mbases') < 30000).filter(pl.col('mbases') >= 8000).join(prev5, on="acc", how="anti")
batch12_possible.shape[0], batch12_possible.sample(5), batch12_possible.select(pl.col('mbases')).sum()[0,0] /1e9

(146031,
 shape: (5, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ SRR25545746 ┆ 2023-08-07T00:00:00+00:00 ┆ 10819  ┆ 3472   │
 │ ERR4869809  ┆ 2022-09-12T00:00:00+00:00 ┆ 21604  ┆ 6819   │
 │ SRR27421869 ┆ 2024-01-05T00:00:00+00:00 ┆ 29400  ┆ 8813   │
 │ ERR11570828 ┆ 2023-09-16T00:00:00+00:00 ┆ 9300   ┆ 3052   │
 │ SRR26912380 ┆ 2024-11-21T00:00:00+00:00 ┆ 11389  ┆ 3455   │
 └─────────────┴───────────────────────────┴────────┴────────┘,
 1.957546801)

In [24]:
# batch12_possible.sample(fraction=1, shuffle=True).select('acc').write_csv('runlists/batch12_8_to_30gbp.txt', include_header=False)

# batch 13 - 30gbp and above

In [25]:
batch13_possible = df.filter(pl.col('mbases') >= 30000).join(prev5, on="acc", how="anti")
batch13_possible.shape[0], batch13_possible.sample(5), batch13_possible.select(pl.col('mbases')).sum()[0,0] /1e9

(16011,
 shape: (5, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ SRR14710884 ┆ 2023-08-17T00:00:00+00:00 ┆ 34403  ┆ 10982  │
 │ SRR9031913  ┆ 2019-05-09T00:00:00+00:00 ┆ 41029  ┆ 13972  │
 │ SRR26046843 ┆ 2023-09-13T00:00:00+00:00 ┆ 53577  ┆ 18673  │
 │ SRR23892488 ┆ 2024-03-01T00:00:00+00:00 ┆ 54030  ┆ 19883  │
 │ SRR15377965 ┆ 2021-09-01T00:00:00+00:00 ┆ 50382  ┆ 14479  │
 └─────────────┴───────────────────────────┴────────┴────────┘,
 0.798983057)

In [26]:
# batch13_possible.sample(fraction=1, shuffle=True).select('acc').write_csv('runlists/batch13_30gbp_plus.txt', include_header=False)

# batch 14 - 30gbp and above retrying failed ones

In [35]:
# ➜  date; rclone ls s3:woodcrob-sandpiper-us-east-1/unannotated6/ >unannotated6.finished.ls.txt
# Mon 24 Mar 2025 09:27:23 AEST
# ➜  wc -l unannotated6.finished.ls.txt
#14262 unannotated6.finished.ls.txt
extern.run('cat  unannotated6.finished.ls.txt |grep RR |sed "s/  */\t/g" >/tmp/runs')
batch13_finished_samples = pl.read_csv("/tmp/runs", has_header=False, separator='\t')
batch13_finished_samples.columns = ["null","size","acc1"]
batch13_finished_samples.drop_in_place("null")
batch13_finished_samples = batch13_finished_samples.with_columns(pl.col("acc1").str.split('.').list.get(0).alias("acc"))
# batch13_finished_samples.columns = ["acc"]
batch13_finished_samples.shape, batch13_finished_samples[:4], batch13_finished_samples.filter(pl.col("size") == 0).shape

((14262, 3),
 shape: (4, 3)
 ┌─────────┬─────────────────────────────────┬───────────┐
 │ size    ┆ acc1                            ┆ acc       │
 │ ---     ┆ ---                             ┆ ---       │
 │ i64     ┆ str                             ┆ str       │
 ╞═════════╪═════════════════════════════════╪═══════════╡
 │ 2925000 ┆ DRR046408.unannotated.singlem.… ┆ DRR046408 │
 │ 5731055 ┆ DRR046817.unannotated.singlem.… ┆ DRR046817 │
 │ 0       ┆ DRR046818.unannotated.singlem.… ┆ DRR046818 │
 │ 2753842 ┆ DRR046819.unannotated.singlem.… ┆ DRR046819 │
 └─────────┴─────────────────────────────────┴───────────┘,
 (1325, 3))

In [36]:
batch14_possible = df.join(batch13_finished_samples.filter(pl.col("size") == 0), on="acc", how="inner")
batch14_possible.shape, batch14_possible[:4]

((1325, 6),
 shape: (4, 6)
 ┌─────────────┬───────────────────────────┬────────┬────────┬──────┬───────────────────────────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes ┆ size ┆ acc1                          │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    ┆ ---  ┆ ---                           │
 │ str         ┆ str                       ┆ i64    ┆ i64    ┆ i64  ┆ str                           │
 ╞═════════════╪═══════════════════════════╪════════╪════════╪══════╪═══════════════════════════════╡
 │ SRR9008757  ┆ 2019-05-04T00:00:00+00:00 ┆ 68571  ┆ 20772  ┆ 0    ┆ SRR9008757.unannotated.single │
 │             ┆                           ┆        ┆        ┆      ┆ m…                            │
 │ SRR14689476 ┆ 2021-12-30T00:00:00+00:00 ┆ 60033  ┆ 17387  ┆ 0    ┆ SRR14689476.unannotated.singl │
 │             ┆                           ┆        ┆        ┆      ┆ e…                            │
 │ SRR30481250 ┆ 2024-08-30T00:00:00+00:00 ┆ 120353 ┆ 3

In [ ]:
# batch14_possible.sample(fraction=1).select('acc').write_csv('runlists/batch14_30gbp_plus_retries.txt', include_header=False)